# FRC 2026 Las Vegas Regional – Pre-Scouting Tool

**Objective:** Retrieve all available performance statistics (OPR, DPR, CCWM) for every team attending the 2026 Las Vegas Regional (`2026nvlv`), based on their most recent prior 2026 competition. Teams are ranked by OPR (Offensive Power Rating).

**Data Source:** [The Blue Alliance API v3](https://www.thebluealliance.com/apidocs/v3)

In [15]:
import requests
import pandas as pd
import numpy as np
import time
import os
from dotenv import load_dotenv

load_dotenv()

True

## Configuration

Set your TBA API key below. Get one at https://www.thebluealliance.com/account

In [16]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
TBA_API_KEY   = os.environ["TBA_API_KEY"]  # Loaded from .env file
TARGET_EVENT  = "2026nvlv"                 # 2026 Las Vegas Regional
CURRENT_YEAR  = 2026
OUR_TEAM      = 2375                       # Exclude our own team from scouting
# ─────────────────────────────────────────────────────────────────────────────

# ── SCORE BREAKDOWN FIELDS TO COLLECT ────────────────────────────────────────
# Maps display column name → TBA COPRS key
# NOTE: Hub fields use space-separated names in the API; other fields use camelCase.
SCORE_FIELDS = {
    "Hub Auto Fuel Count":          "Hub Auto Fuel Count",
    "Hub Teleop Fuel Count":        "Hub Teleop Fuel Count",
    "Hub Endgame Fuel Count":       "Hub Endgame Fuel Count",
    "Hub Total Fuel Count":         "Hub Total Fuel Count",
    "Hub Transition Fuel Count":    "Hub Transition Fuel Count",
    "Hub Shift 1 Fuel Count":       "Hub Shift 1 Fuel Count",
    "Hub Shift 2 Fuel Count":       "Hub Shift 2 Fuel Count",
    "Hub Shift 3 Fuel Count":       "Hub Shift 3 Fuel Count",
    "Hub Shift 4 Fuel Count":       "Hub Shift 4 Fuel Count",
    "Hub First Active Shift Count": "Hub First Active Shift Count",
    "Hub Second Active Shift Count":"Hub Second Active Shift Count",
    "Hub Uncounted":                "Hub Uncounted",
    "Total Auto Points":            "totalAutoPoints",
    "Total Teleop Points":          "totalTeleopPoints",
    "Endgame Tower Points":         "endGameTowerPoints",
    "Total Tower Points":           "totalTowerPoints",
    "Energized Achieved":           "energizedAchieved",
    "Supercharged Achieved":        "superchargedAchieved",
    "Minor Foul Count":             "minorFoulCount",
    "Major Foul Count":             "majorFoulCount",
    "Foul Points":                  "foulPoints",
    "RP":                           "rp",
    "Total Points":                 "totalPoints",
}
# ─────────────────────────────────────────────────────────────────────────────

BASE_URL = "https://www.thebluealliance.com/api/v3"
HEADERS  = {"X-TBA-Auth-Key": TBA_API_KEY}


def tba_get(endpoint: str, retries: int = 3, backoff: float = 2.0):
    """GET wrapper with basic retry / rate-limit handling."""
    url = f"{BASE_URL}{endpoint}"
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:          # rate limited
                wait = backoff * (attempt + 1)
                print(f"Rate limited – waiting {wait}s before retry...")
                time.sleep(wait)
            else:
                print(f"HTTP {resp.status_code} for {url}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request error ({attempt+1}/{retries}): {e}")
            time.sleep(backoff)
    return None

print("Configuration loaded.")

Configuration loaded.


## Step 1 – Fetch Teams Attending the Vegas Regional

In [17]:
teams_data = tba_get(f"/event/{TARGET_EVENT}/teams")

if not teams_data:
    raise RuntimeError("Failed to fetch team list. Check your API key and event key.")

teams = [
    {"team_key": t["key"], "team_number": t["team_number"], "team_name": t["nickname"]}
    for t in teams_data
    if t["team_number"] != OUR_TEAM
]

print(f"Found {len(teams)} teams to scout for {TARGET_EVENT} (excluding Team {OUR_TEAM}).")
teams[:5]   # preview

Found 44 teams to scout for 2026nvlv (excluding Team 2375).


[{'team_key': 'frc1011', 'team_number': 1011, 'team_name': 'CRUSH'},
 {'team_key': 'frc10183', 'team_number': 10183, 'team_name': 'ARC'},
 {'team_key': 'frc10186', 'team_number': 10186, 'team_name': 'feenx'},
 {'team_key': 'frc10903', 'team_number': 10903, 'team_name': 'The Ionizers'},
 {'team_key': 'frc10905', 'team_number': 10905, 'team_name': 'Team LOVE'}]

## Step 2 – Look Up Each Team's Most Recent Prior 2026 Event & Stats (OPR + Component OPRs)

In [18]:
# Per-event caches – populated on first access, reused for subsequent teams
_opr_cache  = {}
_coprs_cache = {}

# Only consider prior events from week 5 or earlier (some teams' last event is week 7)
MAX_PRIOR_WEEK = 5


def get_most_recent_prior_event(team_key: str, target_event_key: str, year: int):
    """
    Returns the most recently completed 2026 event (other than the target)
    for a given team, filtered to week 5 or earlier, sorted by end_date descending.
    Returns None if the team has no qualifying prior events.
    """
    events = tba_get(f"/team/{team_key}/events/{year}")
    if not events:
        return None

    prior = [
        e for e in events
        if e.get("key") != target_event_key
        and e.get("end_date")
        and e.get("week") is not None
        and e["week"] < MAX_PRIOR_WEEK
    ]

    if not prior:
        return None

    prior.sort(key=lambda e: e["end_date"], reverse=True)
    return prior[0]


def get_event_oprs(event_key: str):
    """Fetch (and cache) the OPR data dict for an event."""
    if event_key not in _opr_cache:
        _opr_cache[event_key] = tba_get(f"/event/{event_key}/oprs") or {}
    return _opr_cache[event_key]


def get_event_coprs(event_key: str):
    """Fetch (and cache) the Component OPR data dict for an event."""
    if event_key not in _coprs_cache:
        _coprs_cache[event_key] = tba_get(f"/event/{event_key}/coprs") or {}
    return _coprs_cache[event_key]


def get_team_stats(team_key: str, event_key: str):
    """
    Returns a dict with OPR and all SCORE_FIELDS values for a team at an event.
    OPR comes from /oprs; the rest come from /coprs.
    """
    # OPR
    opr_data = get_event_oprs(event_key)
    opr_val  = opr_data.get("oprs", {}).get(team_key)

    # Component OPRs
    coprs_data = get_event_coprs(event_key)
    stats = {"OPR": opr_val if opr_val is not None else np.nan}
    for col, api_key in SCORE_FIELDS.items():
        val = coprs_data.get(api_key, {}).get(team_key)
        stats[col] = val if val is not None else np.nan

    return stats


print("Helper functions defined.")

Helper functions defined.


## Step 3 – Build the Scouting DataFrame

In [19]:
records = []
total   = len(teams)
nan_stats = {"OPR": np.nan}
nan_stats.update({col: np.nan for col in SCORE_FIELDS})

for i, team in enumerate(teams, start=1):
    team_key    = team["team_key"]
    team_number = team["team_number"]
    team_name   = team["team_name"]

    if i % 10 == 0 or i == total:
        print(f"  Processing {i}/{total}: {team_key}")

    prior_event = get_most_recent_prior_event(team_key, TARGET_EVENT, CURRENT_YEAR)

    if prior_event is None:
        record = {
            "Team Number"                    : team_number,
            "Team Name"                      : team_name,
            "Latest Event Played Before Week 6": "N/A",
        }
        record.update(nan_stats)
        records.append(record)
        continue

    event_key  = prior_event["key"]
    event_name = prior_event.get("name", event_key)

    # OPR + Component OPRs (both cached per event)
    stats = get_team_stats(team_key, event_key)

    record = {
        "Team Number"                    : team_number,
        "Team Name"                      : team_name,
        "Latest Event Played Before Week 6": event_name,
    }
    record.update(stats)
    records.append(record)

    time.sleep(0.05)

print("\nDone processing all teams.")

  Processing 10/44: frc3009
  Processing 20/44: frc6479
  Processing 30/44: frc8387
  Processing 40/44: frc9426
  Processing 44/44: frc991

Done processing all teams.


## Step 4 – Rank & Display Results

In [20]:
df = pd.DataFrame(records)

# Sort: numeric OPR descending, NaN pushed to bottom
df_sorted = df.sort_values(by="OPR", ascending=False, na_position="last").reset_index(drop=True)
df_sorted.index += 1   # 1-based rank
df_sorted.index.name = "Rank"

# Format all stat columns: 2 decimal places or 'N/A'
stat_cols = ["OPR"] + list(SCORE_FIELDS.keys())
df_display = df_sorted.copy()
for col in stat_cols:
    df_display[col] = df_display[col].apply(
        lambda x: f"{x:.2f}" if pd.notna(x) else "N/A"
    )

pd.set_option("display.max_rows",    None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width",       240)

print(f"2026 Las Vegas Regional Pre-Scouting – Rankings by OPR ({len(df_display)} teams)\n")
df_display

2026 Las Vegas Regional Pre-Scouting – Rankings by OPR (44 teams)



,Team Number,Team Name,Latest Event Played Before Week 6,OPR,Hub Auto Fuel Count,Hub Teleop Fuel Count,Hub Endgame Fuel Count,Hub Total Fuel Count,Hub Transition Fuel Count,Hub Shift 1 Fuel Count,Hub Shift 2 Fuel Count,Hub Shift 3 Fuel Count,Hub Shift 4 Fuel Count,Hub First Active Shift Count,Hub Second Active Shift Count,Hub Uncounted,Total Auto Points,Total Teleop Points,Endgame Tower Points,Total Tower Points,Energized Achieved,Supercharged Achieved,Minor Foul Count,Major Foul Count,Foul Points,RP,Total Points
Rank,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,4421,FORGE Robotics,Canadian Pacific Regional,148.00,22.19,124.28,35.52,146.47,8.19,7.58,33.92,12.85,26.21,41.50,39.06,-0.02,23.53,122.79,-1.48,-0.15,0.71,-0.01,0.15,-0.26,1.68,2.07,148.00
2,8338,Bear Force,Canadian Pacific Regional,32.27,1.37,28.34,10.42,29.71,2.09,13.42,-9.57,21.43,-9.46,3.85,11.98,0.01,4.24,29.77,1.43,4.30,0.13,-0.02,-0.22,0.16,-1.74,0.39,32.27
3,9233,Luminous Robotics Team,İstanbul Regional,27.75,4.64,25.53,7.60,30.17,-0.59,4.52,4.33,5.51,4.15,8.85,9.67,0.35,4.64,24.94,-0.59,-0.59,0.10,-0.01,0.02,-0.10,-1.83,0.22,27.75
4,1011,CRUSH,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
5,10183,ARC,Arizona North Regional,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
6,10186,feenx,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
7,10903,The Ionizers,Idaho Regional,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
8,10905,Team LOVE,Arizona North Regional,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
9,11235,Golden Ratio,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A


## (Optional) Export to CSV

In [21]:
output_path = "2026nvlv_prescouting_stats.csv"
df_display.to_csv(output_path)
print(f"Saved to {output_path}")

Saved to 2026nvlv_prescouting_stats.csv
